In [7]:
import os
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms
import torch
from torch.utils.data import DataLoader


In [8]:
class BiPlanarXrayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for pid in sorted(os.listdir(root_dir)):
            for side in ['left', 'right']:
                la_path = os.path.join(root_dir, pid, 'DRR', side, 'la.png')
                pa_path = os.path.join(root_dir, pid, 'DRR', side, 'pa.png')
                if os.path.exists(la_path) and os.path.exists(pa_path):
                    self.samples.append((la_path, pa_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        la_path, pa_path = self.samples[idx]
        la_img = Image.open(la_path).convert('L')
        pa_img = Image.open(pa_path).convert('L')

        if self.transform:
            la_img = self.transform(la_img)
            pa_img = self.transform(pa_img)

        return la_img, pa_img


In [9]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.GaussianBlur(kernel_size=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


In [10]:
dataset = BiPlanarXrayDataset(root_dir='../data/processed', transform=transform)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4)


In [11]:
import timm
import torch.nn as nn

backbone = timm.create_model("convnextv2_tiny", pretrained=True, num_classes=0)  # No classification head
projection_head = nn.Sequential(
    nn.Linear(768, 512),
    nn.ReLU(),
    nn.Linear(512, 128)
)
model = nn.Sequential(backbone, projection_head)


In [12]:
torch.save(backbone.state_dict(), '../src/models/convnextv2_finetuned_xray.pth')